# Problem 050:  Consecutive Prime Sum

> The prime $41$, can be written as the sum of six consecutive primes:
>
> $$41 = 2 + 3 + 5 + 7 + 11 + 13.$$
>
> This is the longest sum of consecutive primes that adds to a prime below one-hundred.
>
> The longest sum of consecutive primes below one-thousand that adds to a prime, contains $21$ terms, and is equal to $953$.
>
> Which prime, below one-million, can be written as the sum of the most consecutive primes?

## Solution $\mathcal O\left(N \ln \ln N\right)$

A first pass of the problem could just find all the prime numbers using a sieve up to $N$, the largest sum allowed.  For efficiency, the prime generator will pass back the list of primes and the sieve.  The list of primes is used to generate the sums, and the sieve is used to identify if the sums are themselves prime.

To generate the sums, a starting index is looped through, and the sum is taken until surpassing the target $N$.  The primes are added in pairs, since every prime greater than $2$ is odd and the sum must be odd, too.  To expediate the loop, note that the maximum number of terms added before surpassing $N$ will at some point be less than the previously found mnaximum, at which point the calculation is done.  Also, there is nothing limiting there to be just one solution, in fact $N=10^7$ yields two solutions, so the return value is a `list[int]` as opposed to a single `int`.

For time scaling, consider how many primes are needed to reach $N$ starting with $2$.  The MathWorld page on [Prime Sums](https://mathworld.wolfram.com/PrimeSums.html) gives the following asymptotics:
$$\sum_{k=1}^n p_k \sim \frac{1}{2}n^2\ln n$$
Inverting this gives the number of terms required to reach $N$:
$$n \sim 2\sqrt{\frac{N}{\ln N}}$$
The next thing to consider is how often this sum of consecutive primes will lead to a prime number.  The OEIS page [OEIS A013916](https://oeis.org/A013916) gives nothing conclusive to the question, but suggests that the frequency occurs similarly to the distribution of the primes themselves.  This suggests that the first pass through the sum starting with $2$ will greatly restrict how many other starting values need to be consdiered to within about $\ln N$.  Thus, with these arguements, the time scaling of checking the sums is $\mathcal O(\sqrt{N\ln N})$.

So after all this fussing of the timescaling associated with the sums, the sieve for prime generation is the slowest part, and the overall timescaling for the entire code is $\mathcal O(N\ln\ln N)$.

In [4]:
%%time

def sieve_of_eratosthenes(num: int) -> (list[int], list[int]):
    sieve = [True] * ((num-1) // 2)
    pmax = int(num**0.5)//2
    for p in range(pmax):
        if sieve[p]:
            k = 2 * p + 3
            l = ((num - 1) // 2 - k**2 // 2) // k + 1
            sieve[k**2//2-1::k] = [False] * l
    return sieve, [2] + [2*i+3 for i in range((num-1)//2) if sieve[i]]
    
def p050(N: int) -> list[int]:
    sieve, primes = sieve_of_eratosthenes(N)

    nmax = 1
    n = 1
    val = []
    for i in range(len(primes)-2):
        if n < nmax:
            break
        if i:
            n = 1
            s = primes[i]
        else:
            n = 2
            s = 5
        while s < N:
            if n >= nmax and sieve[s//2-1]:
                if n == nmax:
                    val.append(s)
                else:
                    val = [s]
                    nmax = n
            s += primes[i+n] + primes[i+n+1]
            n += 2
                
    return val

N = 1_000_000
ans = p050(N)

print(ans)

[997651]
CPU times: user 54.3 ms, sys: 8.79 ms, total: 63 ms
Wall time: 62.1 ms


## Solution $\mathcal O\left(\sqrt N (\ln N)^{3/2}\right)$

Since most of the time for the above code is in the prime generation, what speed up can be made by not finding all primes up to $N$?  The trade off is that a significantly less number of primes need to be found, but more calculation needs to be made to determine if the sums are prime or not.

The first barrier to overcome is to determine how many primes need to be generated.  What we want is the number $x$ such that the sum over all primes less than $x$ is close to $N$.  Furthermore, we want to make sure that the value $x$ overestimates the target, since some extra primes might be needed when calculating sums not starting at $2$ and due to the fact that these equations are asymptotic relations and thus not exact.

The answers in this [Stack Exchange](https://math.stackexchange.com/questions/623872/what-is-the-sum-of-the-prime-numbers-up-to-a-prime-number-n) were helpful in developing the ideas in the following discussion.  The asymptotic relation of the sum of primes less than $x$ is:
$$\sum_{p\le x} p \sim \frac{x^2}{2\ln x}$$
This relation consistently underestimates the actual sum, so inverting the relation would overestimate the $x$ that we need, as desired.  But instead of inverting the equation and having to introduce the Lambert-W function, estimate $\ln x \lesssim \ln N$ and then invert giving:
$$x \simeq \sqrt{2N\ln N}.$$
This approximation gives a robust enough value to get enough primes to answer the question.  Perhaps not a rigorous explanation, but an empirical one.

The only other part of the code that is different is the `is_prime` calculation of the sums, which is just trial division of all primes s.t. $p^2 < s$.

The discussion of the time scaling from the first code now becomes relevant here, where the number of sums being checked is of order $\sqrt{N\ln N}$.  For each of these sums, the `is_prime` calculation for $s$ will be of size $\ln s$, thus giving the quoted scaling.

Better yet, the `is_prime` is not used on all the sums gone over, since the number of terms must be more than the previous maximum value, so the scaling is actually better than this analysis suggests.  Regardless, this version of the code can run for values like $10^{11}$ in less than a minute.

In [22]:
%%time

from math import log

def is_prime(N, primes):
    for p in primes:
        if p**2 > N:
            break
        if not N % p:
            return False
    return True

def sieve_of_eratosthenes(num: int) -> list[int]:
    sieve = [True] * ((num-1) // 2)
    pmax = int(num**0.5)//2
    for p in range(pmax):
        if sieve[p]:
            k = 2 * p + 3
            l = ((num - 1) // 2 - k**2 // 2) // k + 1
            sieve[k**2//2-1::k] = [False] * l
    return [2] + [2*i+3 for i in range((num-1)//2) if sieve[i]]
    
def p050(N: int) -> list[int]:
    primes = sieve_of_eratosthenes(int((2*N*log(N))**0.5)+1)

    nmax = 1
    n = 1
    val = []
    for i in range(len(primes)-2):
        if n < nmax:
            break
        if i:
            n = 1
            s = primes[i]
        else:
            n = 2
            s = 5
        while s < N:
            if n >= nmax and is_prime(s, primes):
                if n == nmax:
                    val.append(s)
                else:
                    val = [s]
                    nmax = n
            s += primes[i+n] + primes[i+n+1]
            n += 2
    return val

N = 1_000_000
ans = p050(N)

print(ans)

[997651]
CPU times: user 3.14 ms, sys: 984 μs, total: 4.12 ms
Wall time: 5.06 ms
